# Findings 

- ResNet's forward method has an `adaptive_avg_pool2d` followed by `torch.flatten`. Looking at the actual ResNet source

- torch.flatten isn't a module — it's a function call inside the forward method, so it doesn't show up when you print the model.
The printed output only shows nn.Module attributes (like Conv2d, BatchNorm2d, AdaptiveAvgPool2d).

- To see the flatten, you'd need to look at the actual source code of ResNet._forward_impl.

In [1]:
import torch 
import torch.nn as nn 
import torchvision

In [2]:
model = torchvision.models.resnet18()
model 

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [3]:
print(model.fc)     # The final layer is a classification layer. 
print(model.conv1)  # The very first layer 
print(model.maxpool)

Linear(in_features=512, out_features=1000, bias=True)
Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)


### ResNet18 

- why `self.features_dim` exist here ? 

In [4]:
class ResNet18(nn.Module): 
    def __init__(self): 
        super().__init__()
        self.backbone = torchvision.models.resnet18()
        self.backbone.fc = nn.Identity()
        self.backbone.conv1 = nn.Conv2d(
            3, 64, kernel_size=3,stride=1,padding=2,bias=False
        )
        self.backbone.maxpool = nn.Identity()
        self.features_dim = 512 # a convenient way to store the output dimesion size.[used later].

    def forward(self,x): 
        return self.backbone(x)
    
backbone = ResNet18()
backbone(torch.randn(1,3,32,32)).shape


torch.Size([1, 512])

### ImageSSL 

In [5]:
class ImageSSL(nn.Module):
    def __init__(
        self,backbone, features_dim, proj_hidden_dim=2048,proj_output_dim = 2048
        ):

        super().__init__()
        self.backbone = backbone
        self.features_dim = features_dim

        # Projector 
        self.projector = nn.Sequential(
            nn.Linear(features_dim,proj_hidden_dim),
            nn.BatchNorm1d(proj_hidden_dim),
            nn.ReLU(),
            nn.Linear(proj_hidden_dim,proj_hidden_dim),
            nn.BatchNorm1d(proj_hidden_dim),
            nn.ReLU(),
            nn.Linear(proj_hidden_dim,proj_output_dim)
        )

    def forward(self,x): 
        features = self.backbone(x)
        projections = self.projector(features)
        return features,projections

In [6]:
model = ImageSSL(
    backbone,
    features_dim= backbone.features_dim,
    proj_hidden_dim= 2048, # cfgs.model.proj_hidden_dim
    proj_output_dim= 2048  # cfgs.model.proj_output_dim
    )
model

ImageSSL(
  (backbone): ResNet18(
    (backbone): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): Identity()
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [7]:
features, projections = model(torch.randn(2,3,32,32))
print(features.shape)
print(projections.shape)


torch.Size([2, 512])
torch.Size([2, 2048])


- `features` : used for linear prob 
- `projections` : used for the VICReg loss (training)

### Linear Probe  

1. Why "probe":

It comes from the idea of probing — like a scientist poking at something to see what's inside. You freeze the backbone and attach a simple linear layer to probe whether the learned features actually contain useful information (like class identity). If a linear layer can classify well from the features, the features must be good.

2. Why it exists in this architecture:

VICReg is self-supervised — it has no labels. So the VICReg loss alone can't tell you if the model is learning anything useful. The linear probe is a monitoring tool during training. It answers: "at epoch N, how good are these features for classification?"

In [8]:
class LinearProbe(nn.Module): 
    """Linear probe classifier for evaluation representations."""
    def __init__(self,features_dim,num_classes):
        super().__init__()
        self.classifier = nn.Linear(features_dim,num_classes)

    def forward(self,x): 
        return self.classifier(x)

linear_probe = LinearProbe(features_dim=512,num_classes=10)
linear_probe(torch.randn(1,512)).shape

torch.Size([1, 10])

The key is .detach() — the linear probe's gradients never reach the backbone. 
So the backbone is trained only by VICReg, and the probe just tells you how well it's doing. Without it, 
you'd be training blindly with no way to track progress during training.